In [1]:
import pandas as pd
import numpy as np
import sys

print("Python version:", sys.version)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

Python version: 3.12.4 (tags/v3.12.4:8e8a4ba, Jun  6 2024, 19:30:16) [MSC v.1940 64 bit (AMD64)]
pandas: 2.3.3
numpy: 2.2.4


In [2]:
import subprocess
subprocess.run(["pip", "install", "entsoe-py", "pyarrow"], check=True)

CompletedProcess(args=['pip', 'install', 'entsoe-py', 'pyarrow'], returncode=0)

In [3]:
from entsoe import EntsoePandasClient
import pyarrow
import requests

print("entsoe-py:", "OK")
print("pyarrow:", pyarrow.__version__)
print("requests:", requests.__version__)

entsoe-py: OK
pyarrow: 24.0.0
requests: 2.32.3


In [4]:
import os

folders = [
    "spark_spread_project/data/raw",
    "spark_spread_project/data/clean",
    "spark_spread_project/src",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"Created: {folder}")

print("\nFolder structure ready.")

Created: spark_spread_project/data/raw
Created: spark_spread_project/data/clean
Created: spark_spread_project/src

Folder structure ready.


In [5]:
import pandas as pd

# Look at the first 3 rows of each file
for filename in ["Germany.csv", "ttf_gas_raw.csv", "carbon_ets_raw.csv"]:
    path = f"spark_spread_project/data/raw/{filename}"
    df = pd.read_csv(path, nrows=3)
    print(f"\n{'='*50}")
    print(f"FILE: {filename}")
    print(f"Columns: {list(df.columns)}")
    print(df.head(3))


FILE: Germany.csv
Columns: ['Country', 'ISO3 Code', 'Datetime (UTC)', 'Datetime (Local)', 'Price (EUR/MWhe)']
   Country ISO3 Code       Datetime (UTC)     Datetime (Local)  \
0  Germany       DEU  2015-01-01 00:00:00  2015-01-01 01:00:00   
1  Germany       DEU  2015-01-01 01:00:00  2015-01-01 02:00:00   
2  Germany       DEU  2015-01-01 02:00:00  2015-01-01 03:00:00   

   Price (EUR/MWhe)  
0             22.34  
1             22.34  
2             22.34  

FILE: ttf_gas_raw.csv
Columns: ['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']
         Date   Price    Open    High     Low   Vol. Change %
0  29/11/2024  47.350  46.450  47.945  46.235  0.54K    1.23%
1  28/11/2024  46.775  47.480  47.480  46.640  0.13K    0.03%
2  27/11/2024  46.762  46.535  46.800  46.000  0.09K   -0.96%

FILE: carbon_ets_raw.csv
Columns: ['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']
         Date  Price   Open   High    Low   Vol. Change %
0  12/29/2023  77.98  78.00  78.61  77.28

In [6]:
for filename, datecol in [
    ("Germany.csv",       "Datetime (UTC)"),
    ("ttf_gas_raw.csv",   "Date"),
    ("carbon_ets_raw.csv","Date"),
]:
    path = f"spark_spread_project/data/raw/{filename}"
    df = pd.read_csv(path)
    df[datecol] = pd.to_datetime(df[datecol])
    print(f"\nFILE: {filename}")
    print(f"  Rows     : {len(df):,}")
    print(f"  From     : {df[datecol].min()}")
    print(f"  To       : {df[datecol].max()}")


FILE: Germany.csv
  Rows     : 99,140
  From     : 2015-01-01 00:00:00
  To       : 2026-04-23 21:00:00

FILE: ttf_gas_raw.csv
  Rows     : 1,790
  From     : 2017-10-23 00:00:00
  To       : 2024-11-29 00:00:00

FILE: carbon_ets_raw.csv
  Rows     : 772
  From     : 2021-01-04 00:00:00
  To       : 2023-12-29 00:00:00


C:\Users\imman\AppData\Local\Temp\ipykernel_16420\136390734.py:8: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[datecol] = pd.to_datetime(df[datecol])


In [7]:
df = pd.read_csv("spark_spread_project/data/raw/ttf_gas_raw.csv", nrows=3)
print(df.columns.tolist())
print(df.head(3))

['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']
         Date   Price    Open    High     Low   Vol. Change %
0  29/11/2024  47.350  46.450  47.945  46.235  0.54K    1.23%
1  28/11/2024  46.775  47.480  47.480  46.640  0.13K    0.03%
2  27/11/2024  46.762  46.535  46.800  46.000  0.09K   -0.96%


In [8]:
# ── Constants ─────────────────────────────────────────────────────
MMBtu_TO_MWh = 3.41214   # 1 MMBtu = 3.41214 MWh (thermal)

# Approximate annual average EUR/USD exchange rates
FX = {2021: 1.183, 2022: 1.053, 2023: 1.082}

# ── 1. Power ──────────────────────────────────────────────────────
power = pd.read_csv("spark_spread_project/data/raw/Germany.csv")
power["price_timestamp"] = pd.to_datetime(power["Datetime (UTC)"])
power = power.rename(columns={"Price (EUR/MWhe)": "power_eur_mwh"})
power = power[["price_timestamp", "power_eur_mwh"]]

# Filter to our study period
power = power[
    (power["price_timestamp"] >= "2021-01-01") &
    (power["price_timestamp"] <= "2023-12-31")
].reset_index(drop=True)

print(f"Power rows: {len(power):,}")
print(power.head(2))

Power rows: 26,257
      price_timestamp  power_eur_mwh
0 2021-01-01 00:00:00          49.57
1 2021-01-01 01:00:00          43.51


In [9]:
# ── 2. TTF Gas ────────────────────────────────────────────────────
gas = pd.read_csv("spark_spread_project/data/raw/ttf_gas_raw.csv")
gas["price_timestamp"] = pd.to_datetime(gas["Date"], dayfirst=True)
gas = gas.sort_values("price_timestamp").reset_index(drop=True)

# Convert USD/MMBtu → EUR/MWh thermal
gas["year"] = gas["price_timestamp"].dt.year
gas["fx"]   = gas["year"].map(FX).fillna(1.10)
gas["gas_eur_mwh"] = gas["Price"] / MMBtu_TO_MWh / gas["fx"]

gas = gas[["price_timestamp", "gas_eur_mwh"]]

# Filter to study period
gas = gas[
    (gas["price_timestamp"] >= "2021-01-01") &
    (gas["price_timestamp"] <= "2023-12-31")
].reset_index(drop=True)

print(f"Gas rows: {len(gas):,}")
print(gas.head(2))

Gas rows: 753
  price_timestamp  gas_eur_mwh
0      2021-01-04     4.916313
1      2021-01-05     4.460479


In [10]:
gas_raw = pd.read_csv("spark_spread_project/data/raw/ttf_gas_raw.csv")
gas_raw["price_timestamp"] = pd.to_datetime(gas_raw["Date"], dayfirst=True)
gas_raw = gas_raw.sort_values("price_timestamp")

# Filter to Jan 2021
jan2021 = gas_raw[gas_raw["price_timestamp"] >= "2021-01-01"].head(5)
print(jan2021[["price_timestamp", "Price"]])

    price_timestamp   Price
984      2021-01-04  19.845
983      2021-01-05  18.005
982      2021-01-06  17.565
981      2021-01-07  19.305
980      2021-01-08  20.120


In [11]:
# ── 2. TTF Gas ────────────────────────────────────────────────────
gas = pd.read_csv("spark_spread_project/data/raw/ttf_gas_raw.csv")
gas["price_timestamp"] = pd.to_datetime(gas["Date"], dayfirst=True)
gas = gas.sort_values("price_timestamp").reset_index(drop=True)

# Kaggle dataset is already in EUR/MWh — no conversion needed
gas = gas.rename(columns={"Price": "gas_eur_mwh"})
gas = gas[["price_timestamp", "gas_eur_mwh"]]

# Filter to study period
gas = gas[
    (gas["price_timestamp"] >= "2021-01-01") &
    (gas["price_timestamp"] <= "2023-12-31")
].reset_index(drop=True)

print(f"Gas rows: {len(gas):,}")
print(gas.head(3))
print(f"\nPrice range: min={gas['gas_eur_mwh'].min():.2f}  max={gas['gas_eur_mwh'].max():.2f}  mean={gas['gas_eur_mwh'].mean():.2f}")

Gas rows: 753
  price_timestamp  gas_eur_mwh
0      2021-01-04       19.845
1      2021-01-05       18.005
2      2021-01-06       17.565

Price range: min=15.53  max=339.19  mean=74.10


In [12]:
# ── 3. Carbon ─────────────────────────────────────────────────────
carbon = pd.read_csv("spark_spread_project/data/raw/carbon_ets_raw.csv")
carbon["price_timestamp"] = pd.to_datetime(carbon["Date"], dayfirst=True)
carbon = carbon.sort_values("price_timestamp").reset_index(drop=True)

carbon = carbon.rename(columns={"Price": "carbon_eur_tonne"})
carbon = carbon[["price_timestamp", "carbon_eur_tonne"]]

# Filter to study period
carbon = carbon[
    (carbon["price_timestamp"] >= "2021-01-01") &
    (carbon["price_timestamp"] <= "2023-12-31")
].reset_index(drop=True)

print(f"Carbon rows: {len(carbon):,}")
print(carbon.head(3))
print(f"\nPrice range: min={carbon['carbon_eur_tonne'].min():.2f}  max={carbon['carbon_eur_tonne'].max():.2f}  mean={carbon['carbon_eur_tonne'].mean():.2f}")

Carbon rows: 772
  price_timestamp  carbon_eur_tonne
0      2021-01-04             33.89
1      2021-01-05             33.15
2      2021-01-06             33.83

Price range: min=31.84  max=98.01  mean=73.19


C:\Users\imman\AppData\Local\Temp\ipykernel_16420\3754356552.py:3: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  carbon["price_timestamp"] = pd.to_datetime(carbon["Date"], dayfirst=True)


In [13]:
# ── Merge all three into one hourly DataFrame ─────────────────────

# Create a complete hourly index for our study period
hourly_index = pd.date_range(start="2021-01-01", end="2023-12-31 23:00:00", freq="h")

# Start with power — already hourly
merged = power.set_index("price_timestamp").reindex(hourly_index)

# Forward-fill gas and carbon (daily → hourly)
gas_indexed    = gas.set_index("price_timestamp").reindex(hourly_index).ffill()
carbon_indexed = carbon.set_index("price_timestamp").reindex(hourly_index).ffill()

# Combine
merged["gas_eur_mwh"]      = gas_indexed["gas_eur_mwh"]
merged["carbon_eur_tonne"] = carbon_indexed["carbon_eur_tonne"]

merged.index.name = "price_timestamp"
merged = merged.reset_index()

print(f"Merged rows : {len(merged):,}")
print(f"Columns     : {list(merged.columns)}")
print(f"Date range  : {merged['price_timestamp'].min()} → {merged['price_timestamp'].max()}")
print(f"\nMissing values:")
print(merged.isnull().sum())

Merged rows : 26,280
Columns     : ['price_timestamp', 'power_eur_mwh', 'gas_eur_mwh', 'carbon_eur_tonne']
Date range  : 2021-01-01 00:00:00 → 2023-12-31 23:00:00

Missing values:
price_timestamp      0
power_eur_mwh       23
gas_eur_mwh         72
carbon_eur_tonne    72
dtype: int64


In [14]:
# Forward-fill then backward-fill to handle all missing values
merged["power_eur_mwh"]      = merged["power_eur_mwh"].ffill().bfill()
merged["gas_eur_mwh"]        = merged["gas_eur_mwh"].ffill().bfill()
merged["carbon_eur_tonne"]   = merged["carbon_eur_tonne"].ffill().bfill()

print("Missing values after fill:")
print(merged.isnull().sum())

print(f"\nFinal check:")
print(f"  Power  — min={merged['power_eur_mwh'].min():.2f}  max={merged['power_eur_mwh'].max():.2f}  mean={merged['power_eur_mwh'].mean():.2f}")
print(f"  Gas    — min={merged['gas_eur_mwh'].min():.2f}  max={merged['gas_eur_mwh'].max():.2f}  mean={merged['gas_eur_mwh'].mean():.2f}")
print(f"  Carbon — min={merged['carbon_eur_tonne'].min():.2f}  max={merged['carbon_eur_tonne'].max():.2f}  mean={merged['carbon_eur_tonne'].mean():.2f}")

Missing values after fill:
price_timestamp     0
power_eur_mwh       0
gas_eur_mwh         0
carbon_eur_tonne    0
dtype: int64

Final check:
  Power  — min=-316.40  max=867.85  mean=143.05
  Gas    — min=15.53  max=339.19  mean=73.54
  Carbon — min=31.84  max=98.01  mean=73.19


In [15]:
import os

output_path = "spark_spread_project/data/clean/merged_hourly.csv"
merged.to_csv(output_path, index=False)

# Verify
saved = pd.read_csv(output_path)
print(f"Saved successfully: {output_path}")
print(f"Rows : {len(saved):,}")
print(f"Size : {os.path.getsize(output_path) / 1024:.1f} KB")
print(f"\nFirst row:")
print(saved.head(1))

Saved successfully: spark_spread_project/data/clean/merged_hourly.csv
Rows : 26,280
Size : 1027.6 KB

First row:
       price_timestamp  power_eur_mwh  gas_eur_mwh  carbon_eur_tonne
0  2021-01-01 00:00:00          49.57       19.845             33.89
